# 🏷️ Introduction to Building NLP Models

## 📚 Text Classification Overview & Sentiment Analysis

---

## I. 🗂️ Introduction to Text Classification

### ❓ What is Text Classification?
**Definition:**
The process of categorizing text into predefined labels (e.g., positive/negative sentiment, spam/ham, topics).

**Real-life examples:**
- 📧 Email classification (spam or not)
- 📰 News article categorization (sports, politics, entertainment)

### 🌟 Applications of Text Classification
- 😊 Sentiment analysis (positive, negative, neutral)
- 🏷️ Topic categorization (news, entertainment, sports)
- 🌐 Language detection
- 🚫 Spam detection

### 🛠️ Basic Techniques in Text Classification
- 📊 Bag of Words (BoW)
- 📈 TF-IDF (Term Frequency - Inverse Document Frequency)
- 🧠 Word Embeddings (Word2Vec, GloVe)

### 🤖 Algorithms for Text Classification
- 🐦 Naive Bayes
- 📏 Support Vector Machines (SVM)
- 🧬 Neural Networks (Deep Learning-based approaches like CNNs, RNNs)

---

## II. 😊 Sentiment Analysis Overview

### ❓ What is Sentiment Analysis?
**Definition:**
Analyzing text to determine the sentiment expressed by the writer, typically positive, negative, or neutral.

### 🌟 Applications of Sentiment Analysis
- 🗨️ Analyzing social media posts
- 🛒 Monitoring customer reviews (e.g., Amazon, Yelp)
- 🏷️ Brand and product perception

### ⚠️ Challenges in Sentiment Analysis
- 🤨 Sarcasm, irony
- ❓ Ambiguity in sentences
- 🌍 Language variations (formal vs. informal)

### 🛠️ Approaches to Sentiment Analysis
- 📋 Rule-based approaches: Lexicons (e.g., VADER, AFINN)
- 🤖 Machine learning models: Naive Bayes, SVM, RNNs

---

In [8]:
# import modules
import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB


In [9]:
# Creating data

# https://www.kaggle.com/datasets/kazanova/sentiment140
df = pd.read_csv('sentiment140.csv',
                 encoding='latin-1',
                 names=['polarity', 'id', 'date', 'query', 'user', 'text'])
print(df.head())
print(df.tail())

   polarity          id                          date     query  \
0         0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1         0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2         0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3         0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4         0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  
         polarity          id                          date     query  \
1599995         4  2193601966  Tue Jun 16 08:40:49 PDT 2009  NO_QUERY   
1599996         4  

In [10]:
# Update Polarity
print(df.polarity.value_counts())
df.polarity = df.polarity.replace({4:1})
print(df.polarity.value_counts())

polarity
0    800000
4    800000
Name: count, dtype: int64
polarity
0    800000
1    800000
Name: count, dtype: int64


In [11]:
# Removing columns
df.drop(columns=['id','date','query','user'],inplace=True)
df.head()

,polarity,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [12]:
# Sample data

df = df.sample(n=50000)
print(df.polarity.value_counts())
df.to_csv('sentiment140-sample.csv',index=False)

polarity
0    25008
1    24992
Name: count, dtype: int64


In [13]:
# Load data

data = pd.read_csv('sentiment140-sample.csv', encoding='latin1')

print(data.head())

   polarity                                               text
0         0                     back of phone fell in a lake. 
1         1  Just bought cute clothes and some shoes from n...
2         1       family is gone. on stickam with versaemerge 
3         0  @macmuso got off and waited for 15 mintues (in...
4         1  I'm being made to wash the mondy.. oh dear   l...


In [15]:
# Preprocessing

def preprocess(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())
    filtered_tokens = [lemmatizer.lemmatize(i) for i in tokens if i.isalpha() and i not in stop_words]
    return ' '.join(filtered_tokens)

data['processed_text'] = data['text'].apply(preprocess)
print(data.head())

   polarity                                               text  \
0         0                     back of phone fell in a lake.    
1         1  Just bought cute clothes and some shoes from n...   
2         1       family is gone. on stickam with versaemerge    
3         0  @macmuso got off and waited for 15 mintues (in...   
4         1  I'm being made to wash the mondy.. oh dear   l...   

                                      processed_test  \
0                               back phone fell lake   
1      bought cute clothes shoe nordstorm happy clam   
2                    family gone stickam versaemerge   
3  macmuso got waited mintues rain catch bus back...   
4  made wash mondy oh dear lol blister finger coo...   

                                      processed_text  
0                               back phone fell lake  
1      bought cute clothes shoe nordstorm happy clam  
2                    family gone stickam versaemerge  
3  macmuso got waited mintues rain catch bus b

In [17]:
# Vectorization

vectorizer = TfidfVectorizer(max_features=500)
x = vectorizer.fit_transform(data['processed_text'])
y = data['polarity']

In [18]:
# Model

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

model = MultinomialNB()
model.fit(x_train,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [19]:
# Predictions

y_pred = model.predict(x_test)
print("Accuracy:",accuracy_score(y_test,y_pred))

Accuracy: 0.7101


In [20]:
# Evaluation

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.73      0.68      0.70      5076
           1       0.69      0.74      0.72      4924

    accuracy                           0.71     10000
   macro avg       0.71      0.71      0.71     10000
weighted avg       0.71      0.71      0.71     10000

